# Diffusion sinh ảnh theo class trên CIFAR-10 (portfolio)

Bạn dựng một diffusion model có điều kiện class, train bằng classifier-free guidance, sinh ảnh CIFAR-10 theo yêu cầu ("một con
ngựa", "một chiếc tàu"), và đo chất lượng bằng các chỉ số tái lập được: FID và KID trên feature Inception-v3, cộng độ chính xác class
theo một classifier ResNet-20. Sản phẩm là notebook này, các checkpoint, lưới ảnh và một `REPORT.md` đọc được với người chưa mở code.

**Cách làm bài**
- Năm hàm có chữ ký cố định ở mục 1 được chấm bằng test trên dữ liệu nhỏ. Mọi phần còn lại (U-Net có điều kiện, vòng train, EMA, đánh
  giá) là thiết kế của bạn; khung và gợi ý nằm trong các cell `explore` bên dưới. Dùng lại code của bài Intermediate 03 (U-Net, schedule,
  DDIM, EMA) và module 03 (ResNet-20, đọc CIFAR-10).
- Không dùng `diffusers`, `torch-fidelity`, `scipy` trong năm hàm chấm điểm. Cell `explore` được dùng thư viện để đối chiếu.
- Chạy bằng kernel `ai-gpu`. Cấu hình PC (RTX 3060 12GB) và laptop (GPU 4GB) ở cell cấu hình.
- Chấm: `uv run pytest tracks/ai-engineer/modules/06-generative-models/exercises/advanced/01-class-conditional-diffusion -q`

In [ ]:
# explore: tìm gốc repo, kiểm CIFAR-10 (137 MB, tải một lần bằng env gốc: uv run python data/download_dataset.py cifar10)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CIFAR = ROOT / "data/raw/cifar10"
print(CIFAR, sorted(p.name for p in (CIFAR / "plain_text").iterdir()))

In [ ]:
import io
import math
import time

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
NULL_CLASS = 10

## 1. Năm hàm được chấm

- `drop_labels(y, p_uncond, null_class, generator)`: trả bản sao của `y` (int64, shape `(B,)`) trong đó mỗi phần tử được thay bằng
  `null_class` khi `torch.rand(B, generator=generator) < p_uncond` (một lần `rand` trên CPU, rồi chuyển sang device của `y`). Không sửa `y`
  tại chỗ.
- `cfg_eps(model, x, t, y, w, null_class)`: nhiễu đã trộn $$\tilde\varepsilon = (1 + w)\,\varepsilon_\theta(x, t, y) - w\,\varepsilon_\theta(x, t, \varnothing)$$, model gọi là
  `model(x, t, y)`. Với `w = 0` chỉ cần một lần forward có điều kiện. Gợi ý hiệu năng: gộp nhánh có điều kiện và nhánh rỗng thành một batch
  `2B` (test không bắt buộc).
- `ddim_sample_cfg(model, y, alpha_bars, steps, w, null_class, generator, shape)`: DDIM $$\eta = 0$$ như bài Intermediate 03 (cùng dãy bước
  `linspace(0, T-1, steps).round()`, kẹp $$\hat x_0$$ vào $$[-1, 1]$$ rồi tính lại $$\hat\varepsilon$$), nhưng nhiễu lấy từ `cfg_eps`. $$x_T$$ là một lần
  `torch.randn((len(y), *shape), generator=generator)` trên CPU rồi chuyển sang device của model. Chạy trong `torch.no_grad()`.
- `fid_from_features(a, b)`: FID giữa hai tập feature `(N, d)` và `(M, d)` (tensor), tính bằng float64, trả `float`. Covariance là ước
  lượng không chệch (chia $$N - 1$$, như `torch.cov`). $$\mathrm{tr}\big((\Sigma_a\Sigma_b)^{1/2}\big)$$ tính được bằng `torch.linalg.eigh` mà không cần
  `scipy`: xem gợi ý trong README của bài.
- `kid_from_features(a, b, subsets=50, subset_size=1000, generator=None)`: KID với kernel $$k(u, v) = (u^\top v/d + 1)^3$$, ước lượng **không chệch**
  của MMD² trên mỗi tập con, $$m = \min(\text{subset\_size}, N, M)$$; mỗi tập con lấy `ia = torch.randperm(N, generator=generator)[:m]` rồi
  `ib = torch.randperm(M, generator=generator)[:m]`. Trả `(mean, std)` (std chia $$n$$, như `np.std`) của các giá trị MMD², kiểu `float`.

In [ ]:
def drop_labels(y: torch.Tensor, p_uncond: float, null_class: int, generator: torch.Generator) -> torch.Tensor:
    """Thay nhãn bằng null_class với xác suất p_uncond (một lần rand(B) từ generator trên CPU)."""
    # TODO
    raise NotImplementedError


def cfg_eps(model: nn.Module, x: torch.Tensor, t: torch.Tensor, y: torch.Tensor, w: float, null_class: int) -> torch.Tensor:
    """(1 + w) * eps(x, t, y) - w * eps(x, t, null)."""
    # TODO
    raise NotImplementedError


@torch.no_grad()
def ddim_sample_cfg(model: nn.Module, y: torch.Tensor, alpha_bars: torch.Tensor, steps: int, w: float, null_class: int,
                    generator: torch.Generator, shape: tuple[int, ...] = (3, 32, 32)) -> torch.Tensor:
    """DDIM eta = 0 với classifier-free guidance; trả x_0 (len(y), *shape)."""
    # TODO
    raise NotImplementedError


def fid_from_features(a: torch.Tensor, b: torch.Tensor) -> float:
    """Fréchet distance giữa Gauss khớp với feature a và feature b (float64)."""
    # TODO
    raise NotImplementedError


def kid_from_features(a: torch.Tensor, b: torch.Tensor, subsets: int = 50, subset_size: int = 1000,
                      generator: torch.Generator | None = None) -> tuple[float, float]:
    """KID: MMD² không chệch với kernel đa thức bậc 3, trung bình và độ lệch chuẩn trên các tập con."""
    # TODO
    raise NotImplementedError

## 2. Dữ liệu và cấu hình

Đọc CIFAR-10 như module 03 (parquet, cột `img` là PNG), đưa về $$[-1, 1]$$. Hai cấu hình gợi ý (bản tham chiếu dùng đúng chúng, số đo trong
`SOLUTION.md`):

| | PC (RTX 3060 12GB) | Laptop (GPU 4GB) |
| --- | --- | --- |
| U-Net | ch 64, mults (1, 2, 2, 2), 2 ResBlock mỗi mức, attention ở 8×8 và giữa, dropout 0.1 | như PC |
| Batch, bước | 128, 35 000 bước | 128, 10 000 tới 15 000 bước |
| Tăng tốc | bf16 autocast + `torch.compile` | bf16 autocast, không compile |
| VRAM đỉnh đo được | 1.4 GB (có compile) | 2.6 GB (không compile) |

In [ ]:
# explore: đọc CIFAR-10 vào GPU dưới dạng uint8 (150 MB), chọn cấu hình
import pandas as pd
from PIL import Image

def load_cifar10(split, limit=None):
    df = pd.read_parquet(CIFAR / "plain_text" / f"{split}-00000-of-00001.parquet")
    if limit is not None:
        df = df.iloc[:limit]
    X = np.stack([np.asarray(Image.open(io.BytesIO(c["bytes"])).convert("RGB")) for c in df["img"]])
    return torch.from_numpy(X).permute(0, 3, 1, 2).contiguous(), torch.from_numpy(df["label"].to_numpy(dtype=np.int64, copy=True))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
X_train, y_train = load_cifar10("train")
X_test, y_test = load_cifar10("test")
CONFIG = dict(steps=35_000, batch=128, compile=True)   # laptop 4GB: dict(steps=12_000, batch=128, compile=False)
print(X_train.shape, X_test.shape, DEVICE, CONFIG)

In [ ]:
# explore: TODO U-Net có điều kiện. Gợi ý: U-Net của bài Intermediate 03 + attention ở độ phân giải thấp + nn.Embedding(11, tdim)
# (10 class + nhãn rỗng) cộng vào time embedding. forward(x, t, y) -> nhiễu dự đoán, cùng shape với x.

In [ ]:
# explore: TODO vòng train: ảnh [-1, 1] + lật ngang, drop_labels(p_uncond=0.1), t đều, loss MSE trên nhiễu, AdamW lr 2e-4 + warmup,
# clip grad 1.0, EMA, bf16 autocast; log loss và thời gian mỗi 1 000 bước; lưu checkpoint (trọng số thô + EMA) mỗi 5 000 bước.

In [ ]:
# explore: TODO giám khảo: ResNet-20 train trên CIFAR-10 train (module 03, khoảng 30 epoch, ghi test accuracy), và feature Inception-v3
# của torchvision (pool 2048 chiều, ảnh resize 299, chuẩn hoá ImageNet) cho 10 000 ảnh test. Lưu feature ra đĩa để dùng lại.

In [ ]:
# explore: TODO đánh giá: với mỗi w trong (0, 0.5, 1, 2, 4) và bản không điều kiện, sinh 5 000 ảnh (500 mỗi class, cùng seed) bằng
# ddim_sample_cfg 50 bước, tính FID và KID so với 10 000 ảnh test, class accuracy theo ResNet-20, thời gian; lưu lưới 10x10 ảnh.
# Thêm: FID giữa ảnh train thật và ảnh test thật ở cùng số mẫu (mốc), EMA so với trọng số thô, số bước DDIM, checkpoint sớm.